In [12]:
!pip install pillow

# Mango Disease Detection System
Train a model using TensorFlow, Keras, and EfficientNetB0 transfer learning to detect mango diseases from images.

In [12]:

import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, models, callbacks
from sklearn.utils.class_weight import compute_class_weight

### 1. Data Preprocessing & Augmentation
We split datasets into 80% train and 20% validation using ImageDataGenerator's `validation_split`.

In [13]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
DATA_DIR = 'datasets' # Local relative to 'mango' folder
MODEL_DIR = 'model' # Where to save the model and json labels

# Ensure output directory exists
os.makedirs(MODEL_DIR, exist_ok=True)

# Applying Data Augmentation for training
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    zoom_range=0.2,
    horizontal_flip=True,
    validation_split=0.2 # 80/20 train/val split
)

print("Loading Training Data...")
train_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True
)

print("Loading Validation Data...")
val_gen = datagen.flow_from_directory(
    DATA_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False
)

Loading Training Data...
Found 3200 images belonging to 8 classes.
Loading Validation Data...
Found 800 images belonging to 8 classes.


### 2. Output Class Labels & Handle Imbalance
Mapping class indices to class labels. Computing class weights for imbalanced datasets.

In [14]:
# Extract and save class labels
class_indices = train_gen.class_indices
class_labels = {v: k for k, v in class_indices.items()} # Swap keys and values
labels_path = os.path.join(MODEL_DIR, 'mango_labels.json')

with open(labels_path, 'w') as f:
    json.dump(class_labels, f, indent=4)
print(f"Saved class labels to {labels_path}")

# Calculate class weights to handle imbalanced datasets
classes = train_gen.classes
class_weight_arr = compute_class_weight(
    class_weight='balanced', 
    classes=np.unique(classes), 
    y=classes
)
class_weights = {i: weight for i, weight in enumerate(class_weight_arr)}
print(f"Computed Class Weights: {class_weights}")

Saved class labels to model\mango_labels.json
Computed Class Weights: {0: np.float64(1.0), 1: np.float64(1.0), 2: np.float64(1.0), 3: np.float64(1.0), 4: np.float64(1.0), 5: np.float64(1.0), 6: np.float64(1.0), 7: np.float64(1.0)}


### 3. Setup Transfer Learning Model (EfficientNetB0)
We use EfficientNetB0 for high accuracy and efficiency. Top classification layers are custom designed for our specific classes.

In [15]:
num_classes = len(class_indices)

# Load base model with un-trained custom top layer
base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(IMG_SIZE[0], IMG_SIZE[1], 3))
base_model.trainable = False # Freeze the base model for initial training

# Build the custom model on top
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(num_classes, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ efficientnetb0 (Functional)          │ (None, 7, 7, 1280)          │       4,049,571 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_average_pooling2d_2           │ (None, 1280)                │               0 │
│ (GlobalAveragePooling2D)             │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 1280)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 8)                   │          10,248 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 4,059,819 (15.49 MB)

 Trainable params: 10,248 (40.03 KB)

 Non-trainable params: 4,049,571 (15.45 MB)

### 4. Train the Model
Running initial training with callbacks to prevent overfitting and save best performance checkpoint.

In [16]:
checkpoint_path = os.path.join(MODEL_DIR, 'mango_model.keras')

callbacks_list = [
    callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True),
    callbacks.ModelCheckpoint(filepath=checkpoint_path, monitor='val_accuracy', save_best_only=True, verbose=1)
]

INITIAL_EPOCHS = 4

history = model.fit(
    train_gen,
    epochs=INITIAL_EPOCHS,
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=callbacks_list
)

Epoch 1/4
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 842ms/step - accuracy: 0.1373 - loss: 2.1240
Epoch 1: val_accuracy improved from None to 0.12500, saving model to model\mango_model.keras

Epoch 1: finished saving model to model\mango_model.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 115s 1s/step - accuracy: 0.1394 - loss: 2.1159 - val_accuracy: 0.1250 - val_loss: 2.0892
Epoch 2/4
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 984ms/step - accuracy: 0.1067 - loss: 2.1252
Epoch 2: val_accuracy did not improve from 0.12500
100/100 ━━━━━━━━━━━━━━━━━━━━ 111s 1s/step - accuracy: 0.1128 - loss: 2.1207 - val_accuracy: 0.1250 - val_loss: 2.0975
Epoch 3/4
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 385ms/step - accuracy: 0.1243 - loss: 2.1039
Epoch 3: val_accuracy did not improve from 0.12500
100/100 ━━━━━━━━━━━━━━━━━━━━ 48s 479ms/step - accuracy: 0.1266 - loss: 2.1080 - val_accuracy: 0.1250 - val_loss: 2.0913
Epoch 4/4
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 388ms/step - accuracy: 0.1447 - loss: 2.1075
Epoch 4: val_accuracy did not improve from

### 5. Fine-tuning the Model
Unfreeze base model partly or entirely to adjust features to mango specific leaves and spots using a lower learning rate.

In [18]:
# Unfreeze the base model
base_model.trainable = True

# Freeze some lower blocks if you only want the upper layers unfreezed
# Here we just lower the learning rate significantly for full fine-tuning
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

FINE_TUNE_EPOCHS = 2
TOTAL_EPOCHS = INITIAL_EPOCHS + FINE_TUNE_EPOCHS

# Continue training
history_fine = model.fit(
    train_gen,
    epochs=TOTAL_EPOCHS,
    initial_epoch=history.epoch[-1], # Resume where we left off
    validation_data=val_gen,
    class_weight=class_weights,
    callbacks=callbacks_list
)

Epoch 4/6
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9331 - loss: 0.5560
Epoch 4: val_accuracy improved from 0.26750 to 0.38125, saving model to model\mango_model.keras

Epoch 4: finished saving model to model\mango_model.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 255s 2s/step - accuracy: 0.9366 - loss: 0.5058 - val_accuracy: 0.3812 - val_loss: 1.6928
Epoch 5/6
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9481 - loss: 0.3697
Epoch 5: val_accuracy improved from 0.38125 to 0.67750, saving model to model\mango_model.keras

Epoch 5: finished saving model to model\mango_model.keras
100/100 ━━━━━━━━━━━━━━━━━━━━ 219s 2s/step - accuracy: 0.9519 - loss: 0.3490 - val_accuracy: 0.6775 - val_loss: 1.0479
Epoch 6/6
100/100 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9557 - loss: 0.2710
Epoch 6: val_accuracy did not improve from 0.67750
100/100 ━━━━━━━━━━━━━━━━━━━━ 212s 2s/step - accuracy: 0.9603 - loss: 0.2568 - val_accuracy: 0.1562 - val_loss: 3.7598


### 6. Predict Function (Ready for FastAPI integration)
A utility function to load an image, preprocess it, and predict the disease.

In [20]:
from tensorflow.keras.preprocessing import image

def load_labels(json_path):
    with open(json_path, 'r') as f:
        return json.load(f)

def predict_disease(img_path, model_path=checkpoint_path, labels_path=labels_path):
    # Load Model and Labels
    loaded_model = tf.keras.models.load_model(model_path)
    labels_dict = load_labels(labels_path)
    
    # Prepare Image
    img = image.load_img(img_path, target_size=IMG_SIZE)
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0  # Same `rescale` applied in ImageDataGenerator
    
    # Predict
    predictions = loaded_model.predict(img_array)[0]
    predicted_class_index = np.argmax(predictions)
    confidence = float(predictions[predicted_class_index])
    
    # Make sure index correctly converts to string to match JSON keys
    predicted_class_name = labels_dict[str(predicted_class_index)]
    
    return {
        "disease": predicted_class_name,
        "confidence": round(confidence * 100, 2)
    }

Test standard edge case locally (if you have an image to test with)
test_img = 'datasets/Healthy/some_image.jpg'
if os.path.exists(test_img):
   print(predict_disease(test_img))